# `ferrodata` Walk-through

This notebook is a practical guide for:
- testing the new `ferrodata` package,
- reviewing parsed data structures,
- exporting human-readable CSV files,
- plotting `PUND`, `DHM`, and `Fatigue` data.

It uses the example files in `new ferro/`.

## 0) Import and Setup

If you open this notebook from the project root, the local package import should work directly.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from ferrodata import (
    read_dat,
    get_waveform_tables,
    get_fatigue_result_table,
    export_all_tables_csv,
    plot_pund,
    plot_dhm,
    plot_fatigue,
)

plt.rcParams.update({"figure.dpi": 110})

## 1) Point to Example Files

Edit these paths if your files are in another location.

In [ ]:
base_dir = Path("examples")
example_files = {
    "pund": base_dir / "[example data]PUND.dat",
    "dhm": base_dir / "[example data]DHM.dat",
    "fatigue": base_dir / "[example data]Fatigue.dat",
}

for name, path in example_files.items():
    print(f"{name:8s} -> {path} | exists={path.exists()}")

## 2) Parse Files

`read_dat(...)` returns a `MeasurementFile` object with:
- `measurement_type`
- `global_metadata`
- `tables`

In [ ]:
parsed = {name: read_dat(path) for name, path in example_files.items()}

for name, obj in parsed.items():
    print(f" [{name}] type={obj.measurement_type.value}")
    print(f"  global metadata keys: {len(obj.global_metadata)}")
    print(f"  number of tables:      {len(obj.tables)}")

## 3) Inspect Table Structure

Use this helper to quickly inspect one table.

In [ ]:
def inspect_table(table, n_rows=3):
    print(f"Table name: {table.name}")
    print(f"Columns: {len(table.headers)} | Rows: {table.data.shape[0]}")
    print("First 8 metadata items:")
    for i, (k, v) in enumerate(table.metadata.items()):
        if i >= 8:
            break
        print(f"  - {k}: {v}")

    if table.headers:
        print("\nHeaders (first 8):")
        print(table.headers[:8])

    if table.data.size:
        print("\nFirst rows:")
        print(table.data[:n_rows])


inspect_table(parsed["pund"].tables[0])

## 4) Export Tables to CSV

This writes each parsed table to one CSV file.

In [ ]:
export_root = Path("ferrodata_exports")
export_root.mkdir(exist_ok=True)

for name, obj in parsed.items():
    out_dir = export_root / name
    csv_paths = export_all_tables_csv(obj, out_dir, include_metadata=True)
    print(f"{name:8s}: exported {len(csv_paths)} files -> {out_dir}")
    if csv_paths:
        print(f"  first file: {csv_paths[0].name}")

## 5) Plot PUND

Pick one waveform table and plot raw current, switched current density, and PU/ND polarization.

In [ ]:
pund_wave_tables = get_waveform_tables(parsed["pund"])
print("PUND waveform tables:", len(pund_wave_tables))

fig, axes = plot_pund(pund_wave_tables[0], field_unit="MV/cm")
plt.show()

## 6) Plot DHM

Plot voltage, current, and polarization views from one DHM waveform table.

In [ ]:
dhm_wave_tables = get_waveform_tables(parsed["dhm"])
print("DHM waveform tables:", len(dhm_wave_tables))

fig, axes = plot_dhm(dhm_wave_tables[0], field_unit="MV/cm")
plt.show()

## 7) Plot Fatigue Summary

Use `get_fatigue_result_table(...)` to fetch the summary table (cycles vs polarization metrics).

In [ ]:
fatigue_result = get_fatigue_result_table(parsed["fatigue"])
print("Fatigue result table:", fatigue_result.name)

fig, axes = plot_fatigue(fatigue_result)
plt.show()

## 8) Minimal Reusable Workflow

This helper chooses a default plot based on measurement type.

In [ ]:
def quick_plot(dat_path):
    m = read_dat(dat_path)
    print(f"type={m.measurement_type.value} | tables={len(m.tables)}")

    if m.measurement_type.value == "fatigue":
        table = get_fatigue_result_table(m)
        fig, _ = plot_fatigue(table)
    else:
        wave = get_waveform_tables(m)[0]
        if m.measurement_type.value == "pund":
            fig, _ = plot_pund(wave)
        else:
            fig, _ = plot_dhm(wave)

    return m, fig


# Example:
# m, fig = quick_plot(example_files["pund"])
# plt.show()

## Notes

- This walk-through intentionally uses the raw metadata in the file.
- For batch processing, loop over file paths and reuse the same API calls shown above.